# MESSAGEix-Pakistan 
### Current Measures Scenario
In this notebook, we are reading data and building a current measures scenerio.

<img src="https://wit.lums.edu.pk/sites/default/files/inline-images/WIT_Banner.jpg" alt="Girl in a jacket" width="850" height="250">

In [1]:
import os
import sys
from pathlib import Path

# resolve the message_pak root (one level above this notebook's directory)
ROOT = Path(os.path.abspath('')).parent
sys.path.insert(0, str(ROOT))

# fundamental libraries
import ixmp
import message_ix
from message_ix import log

from scripts.calibrate_t_d import model_calibrate
from scripts.reserve_margin import update_reserve_margin
from scripts.add_rooftop_solar_pv import main as add_rooftop_solar_pv

# autoreload modules when changes are applied to them
%load_ext autoreload  
%autoreload all
%reload_ext autoreload
%matplotlib inline

import warnings
warnings.filterwarnings('ignore')

In [2]:
# Confirm root path (message_pak directory)
print('Project root:', ROOT)

Project root: /Users/awais/Documents/GitHub/NEST-Pakistan/MESSAGEix-Pakistan


Create the Scenario

In [3]:
# creating ixmp platform object
new_mp = ixmp.Platform()

# creating a new, empty scenario object
scenario = message_ix.Scenario(
    new_mp, 
    model="MESSAGEix-Pakistan", 
    scenario="current-measures", version="new"
)

2026-06-22 11:42:13,071  INFO at.ac.iiasa.ixmp.Platform:171 - Welcome to the IX modeling platform!
2026-06-22 11:42:13,082  INFO at.ac.iiasa.ixmp.Platform:172 -  connected to database 'jdbc:hsqldb:file:/Users/awais/.local/share/ixmp/localdb/default' (user: ixmp)...


Read Model Data

In [4]:
data_path = ROOT / 'data' / 'MESSAGEix-Pakistan-CurrentMeasures-SSP2.xlsx'
scenario.read_excel(str(data_path), add_units=True, commit_steps=False, init_items=True)

In [7]:
update_reserve_margin(scenario, ["R12_PAK"], contin=0.15)

##### Add Rooftop Solar PV

In [6]:
path_data_RT = ROOT / 'data' / 'VRE_calibration_SSP_dev_v14.xlsx'
add_rooftop_solar_pv(scenario, path_data_RT=path_data_RT, ssp="SSP2", verbose=True)

Working on ... capacity_factor
['node_loc', 'technology', 'year_vtg', 'year_act', 'time']
Working on ... growth_activity_lo
['node_loc', 'technology', 'year_act', 'time']
Working on ... growth_new_capacity_up
['node_loc', 'technology', 'year_vtg']
Working on ... initial_activity_lo
['node_loc', 'technology', 'year_act', 'time']
Working on ... initial_new_capacity_up
['node_loc', 'technology', 'year_vtg']
Working on ... inv_cost
['node_loc', 'technology', 'year_vtg']
Working on ... fix_cost
['node_loc', 'technology', 'year_vtg', 'year_act']
Working on ... technical_lifetime
['node_loc', 'technology', 'year_vtg']
Working on ... relation_total_capacity
['relation', 'node_rel', 'year_rel', 'technology']
Working on ... soft_new_capacity_up
['node_loc', 'technology', 'year_vtg']
Working on ... abs_cost_new_capacity_soft_up
['node_loc', 'technology', 'year_vtg']
Working on ... bound_new_capacity_up
['node_loc', 'technology', 'year_vtg']
Working on ... bound_new_capacity_lo
['node_loc', 'techn

##### Solve the Model

In [8]:
log.info(f"version number before commit(): {scenario.version}")
# scenario.commit("Current measures")
scenario.set_as_default()

# exporting the built model (Scenario) to GAMS with an optional case name
caseName = scenario.model + '__' + scenario.scenario + '__v' + str(scenario.version)

# solve model
scenario.solve(case=caseName)
obj_value = scenario.var("OBJ")["lvl"]
print(f"Objective value: {obj_value}")

com.gams.api.GAMSException: com.gams.api.GAMSException: GAMS system directory is both not specified and not found from your environment!

##### Report the Results

In [ ]:
from report.legacy.iamc_report_hackathon import report
from datetime import datetime
import time

timestamp = datetime.now().strftime('%Y-%m-%d--%H-%M')
start = time.time()
out_dir = ROOT / 'output'
out_dir.mkdir(parents=True, exist_ok=True)
df, path_name = report(mp=new_mp, scen=scenario, out_dir=str(out_dir), out_file_timestamp=timestamp, IDEA_format=False)
end = time.time()
print(f'Reporting done in {end - start:.1f}s → {out_dir}')

processing Table: Resource|Extraction
processing Table: Resource|Cumulative Extraction
processing Table: Primary Energy
processing Table: Primary Energy (substitution method)
processing Table: Final Energy
processing Table: Secondary Energy|Electricity
processing Table: Secondary Energy|Heat
processing Table: Secondary Energy
processing Table: Secondary Energy|Gases
processing Table: Secondary Energy|Solids
processing Table: Emissions|CO2
processing Table: Carbon Sequestration
processing Table: Emissions|BC
processing Table: Emissions|OC
processing Table: Emissions|CO
processing Table: Emissions|N2O
processing Table: Emissions|CH4
processing Table: Emissions|NH3
processing Table: Emissions|Sulfur
processing Table: Emissions|NOx
processing Table: Emissions|VOC
processing Table: Emissions|HFC
HFC125
No objects to concatenate
processing Table: Emissions
processing Table: Emissions
processing Table: Agricultural Demand
module 'report.legacy.default_tables_static' has no attribute 'retr_agr

In [ ]:
# close the connection to the database
new_mp.close_db()